# PAISA OGDC price-only LSTM experiments

This thin Colab/CPU notebook invokes PAISA's reusable experiment runner for the frozen M-005 and M-006 variants. Sequence construction, preprocessing, model training, evaluation, artifact sealing, common-date comparison, and portfolio simulation remain in repository code.

The generated results are an academic simulation and are not financial advice.

## 1. Resolve PAISA and install the optional TensorFlow runtime

Set `PAISA_GIT_REF` before this cell to run a reviewed feature branch or commit in Colab. TensorFlow is installed only for this optional workflow; it is not part of PAISA's default CPU requirements.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Arfa01/PAISA.git"
GIT_REF = os.environ.get("PAISA_GIT_REF", "main")
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    REPO_DIR = Path("/content/PAISA")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    elif not (REPO_DIR / "pyproject.toml").is_file():
        raise RuntimeError(f"Existing path is not a PAISA checkout: {REPO_DIR}")
    origin = subprocess.run(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR,
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if origin.rstrip("/").removesuffix(".git") != REPO_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError(f"Existing checkout has unexpected origin: {origin}")
    dirty = subprocess.run(
        ["git", "status", "--porcelain"], cwd=REPO_DIR,
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError("Existing Colab checkout is dirty; preserve or remove it first.")
    subprocess.run(["git", "fetch", "origin", GIT_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    current = Path.cwd()
    REPO_DIR = current if (current / "pyproject.toml").is_file() else current / "PAISA"
    if not (REPO_DIR / "pyproject.toml").is_file():
        raise RuntimeError(f"Run this notebook from a PAISA checkout: {REPO_DIR}")

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-lstm.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "scripts/run_lstm_price_only_experiments.py", "--help"], check=True)

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR,
    check=True, capture_output=True, text=True,
).stdout.strip()
print("Repository:", REPO_DIR)
print("Resolved commit:", commit)

## 2. Bind immutable baseline evidence and a new run ID

The retained model-training and vertical-slice artifacts are generated local content and are not in a fresh clone. Restore their checksum-verified directories or set the three path variables below to equivalent immutable evidence. Set `PAISA_LSTM_RUN_ID` to a new unused ID; collisions are rejected.

In [ ]:
BASELINE_TRAINING_RUN = Path(os.environ.get(
    "PAISA_BASELINE_TRAINING_RUN",
    "artifacts/model_training/TRN-20261020-191815-624662-nogit",
))
BASELINE_VERTICAL_SLICE = Path(os.environ.get(
    "PAISA_BASELINE_VERTICAL_SLICE",
    "artifacts/vertical_slice/VSL-20261020-OGDC-main-02",
))
PRICES_PATH = Path(os.environ.get(
    "PAISA_PRICES_PATH",
    str(BASELINE_TRAINING_RUN / "frozen_input/OGDC_prices.csv"),
))
OUTPUT_DIR = Path(os.environ.get("PAISA_LSTM_OUTPUT_DIR", "artifacts/lstm_experiments"))
RUN_ID = os.environ.get("PAISA_LSTM_RUN_ID", "").strip()

if not RUN_ID:
    raise RuntimeError("Set PAISA_LSTM_RUN_ID to a new, unused immutable run ID.")
for required_path in (PRICES_PATH, BASELINE_TRAINING_RUN, BASELINE_VERTICAL_SLICE):
    if not required_path.exists():
        raise FileNotFoundError(f"Required immutable input is missing: {required_path}")

print("Prices:", PRICES_PATH)
print("Baseline training run:", BASELINE_TRAINING_RUN)
print("Baseline vertical slice:", BASELINE_VERTICAL_SLICE)
print("New LSTM run ID:", RUN_ID)

## 3. Run the reusable M-005/M-006 experiment

This is the notebook's only experiment call. The repository runner owns deterministic seeds, train-only preprocessing, 20-session sequences, validation-only training decisions, post-freeze test evaluation, artifacts, fair baseline comparison, and independent backtests.

In [ ]:
from paisa.experiments.lstm_runner import run_lstm_price_only_experiments

summary = run_lstm_price_only_experiments(
    prices_path=PRICES_PATH,
    output_dir=OUTPUT_DIR,
    run_id=RUN_ID,
    baseline_training_run=BASELINE_TRAINING_RUN,
    baseline_vertical_slice=BASELINE_VERTICAL_SLICE,
    create_zip=True,
)
print("Completed run:", summary["run_id"])
print("Validation-preferred variant:", summary["validation_preferred_variant"])
print(summary["disclaimer"])

## 4. Read saved leaderboards and authoritative summary

Validation preference, post-freeze LSTM test metrics, and portfolio results remain separate. The same 184-date holdout and M-001 through M-004 outcomes were already exposed by the baseline experiment, so this is repeated evaluation on a shared project holdout—not a globally untouched test. `final_summary.json` is the authoritative result; this notebook only renders its saved evidence.

In [ ]:
import json

import pandas as pd
from IPython.display import display

RUN_DIR = Path(summary["artifact_directory"])
validation_leaderboard = pd.read_csv(RUN_DIR / "validation_leaderboard.csv")
test_leaderboard = pd.read_csv(RUN_DIR / "test_leaderboard.csv")
final_summary = json.loads((RUN_DIR / "final_summary.json").read_text(encoding="utf-8"))

print("Validation leaderboard (selection/reporting evidence)")
display(validation_leaderboard)
print("Post-freeze LSTM test leaderboard (not a selection rule; shared holdout)")
display(test_leaderboard)
print("Authoritative final summary")
display(final_summary)

## 5. Download the immutable archive in Colab

Preserve the complete archive and its checksums. Do not copy individual metrics out of context or describe simulated outputs as financial advice.

In [ ]:
zip_value = summary.get("zip_path")
if zip_value:
    ZIP_PATH = Path(zip_value)
    print("Immutable archive:", ZIP_PATH)
    if IN_COLAB:
        from google.colab import files
        files.download(str(ZIP_PATH))
else:
    print("No ZIP was requested; preserve the complete run directory:", RUN_DIR)